# 🧪 ICU Demand Forecasting - XGBoost
This notebook trains an XGBoost Regressor to predict ICU bed demand based on historical patient flow and hospital metrics.

In [ ]:
from xgboost import XGBRegressor
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

# Load data
df = pd.read_csv("../data/processed/final_hospital_forecasting_dataset.csv")
df['Date'] = pd.to_datetime(df['Date'])

## 1. Feature Engineering

In [ ]:
def extract_date_features(X):
    X['Year'] = X['Date'].dt.year
    X['Month'] = X['Date'].dt.month
    X['Day'] = X['Date'].dt.day
    X['DayOfWeek'] = X['Date'].dt.dayofweek
    return X.drop('Date', axis=1)

X = df.drop('ICU_Demand', axis=1)
y = df['ICU_Demand']

X = extract_date_features(X)
X = pd.get_dummies(X, columns=['service'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 2. Model Training

In [ ]:
model = XGBRegressor(max_depth=3, learning_rate=0.1, n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"Train Score: {model.score(X_train, y_train):.4f}")
print(f"Test Score: {model.score(X_test, y_test):.4f}")

## 3. Evaluation

In [ ]:
preds = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, preds):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")

joblib.dump(model, "../models/icu_xgboost.pkl")